In [216]:
import pandas as pd
import numpy as np
import os

os.makedirs("deductions", exist_ok=True)  # creates it if it doesn't exist


# Set Variables

In [217]:
min_tax_year = 2024
max_tax_year = 2041

max_tax_year_id = max_tax_year % 100


# Tax Years

In [218]:


data = []
for year in range(min_tax_year, max_tax_year + 1):
    tax_year_name = f"{str(year)[2:]}/{str(year + 1)[2:]}"
    tax_year_start = f"06/04/{year}"
    tax_year_end = f"05/04/{year + 1}"
    data.append([tax_year_name, tax_year_start, tax_year_end, year % 100])

tax_years = pd.DataFrame(data, columns=["tax_year_name", "tax_year_start", "tax_year_end", "tax_year_id"])



# For each row, generate every date from tax_year_start to tax_year_end (inclusive), as a list
tax_years["date"] = tax_years.apply(
    lambda row: pd.date_range(start=row["tax_year_start"], end=row["tax_year_end"]), axis=1
)

# "Explode" turns each list of dates into its own separate row,
# duplicating the other columns (like payment_date) across each one
tax_years = tax_years.explode("date")

tax_years.to_csv('deductions/tax_years.csv', index= True)



# Deduction Control
### Income Tax Thresholds

In [219]:
columns = ["tax_year_start", "threshold_one", "threshold_two","threshold_three", "rate_one","rate_two","rate_three", "confirmed"]

data = []

# tax_year_name, threshold_one, threshold_two, threshold three, rate_one, rate_two, rate_three confirmed
data.append(["24/25", 12571, 50271, 125141, 0.20, 0.40, 0.45, True])


income_tax = pd.DataFrame(data, columns=columns)
income_tax["source"] = 'income_tax'

### National Insurance Class 4

In [220]:
columns = ["tax_year_start", "threshold_one", "threshold_two", "rate_one","rate_two", "confirmed"]

data = []

# tax_year_name, threshold_one, threshold_two, rate_one, rate_two, confirmed
data.append(["24/25", 12570, 50271, 0.06, 0.02, True])

ni_class_four = pd.DataFrame(data, columns=columns)
ni_class_four["source"] = 'ni_class_four'

### National Insurance Class 1

In [221]:
columns = ["tax_year_start", "threshold_one", "threshold_two", "rate_one","rate_two", "confirmed"]

data = []

# tax_year_name, threshold_one, threshold_two, rate_one, rate_two, confirmed
data.append(["24/25", 12570, 50271, 0.08, 0.02, True])


ni_class_one = pd.DataFrame(data, columns=columns)
ni_class_one["source"] = 'ni_class_one'

### Student Loans

In [222]:
columns = ["tax_year_start", "threshold_one", "rate_one", "confirmed"]

data = []

# tax_year_name, threshold, rate, confirmed
data.append(["24/25", 24990, 0.09, True])
data.append(["25/26", 26065, 0.09, True])
data.append(["26/27", 26900, 0.09, True])

student_loans = pd.DataFrame(data, columns=columns)
student_loans["source"] = 'income_tax'

# Deduction Calcs

In [223]:
calc_thresholds = pd.concat([income_tax, ni_class_four, ni_class_one, student_loans], ignore_index=False)
calc_thresholds["new_index"]= calc_thresholds.index

calc_thresholds_level_1 = calc_thresholds[["tax_year_start","threshold_one","threshold_two","rate_one","confirmed","source","new_index"]]
calc_thresholds_level_1 = calc_thresholds_level_1.rename(columns={"threshold_one": "lower_threshold","threshold_two": "upper_threshold","rate_one": "rate"})
calc_thresholds_level_1["level"] = 1

calc_thresholds_level_2 = calc_thresholds[["tax_year_start","threshold_two","threshold_three","rate_two","confirmed","source","new_index"]]
calc_thresholds_level_2 = calc_thresholds_level_2.rename(columns={"threshold_two": "lower_threshold","threshold_three": "upper_threshold","rate_two": "rate"})
calc_thresholds_level_2["level"] = 2

calc_thresholds_level_3 = calc_thresholds[["tax_year_start","threshold_three","rate_three","confirmed","source","new_index"]]
calc_thresholds_level_3["upper_threshold"] = 999999
calc_thresholds_level_3 = calc_thresholds_level_3.rename(columns={"threshold_three": "lower_threshold","rate_three": "rate"})
calc_thresholds_level_3["level"] = 3

calc_thresholds = pd.concat([calc_thresholds_level_1, calc_thresholds_level_2, calc_thresholds_level_3], ignore_index=True)
calc_thresholds = calc_thresholds[calc_thresholds["lower_threshold"].notnull()]

# set any values equal to "NA" to null (NaN)
calc_thresholds["upper_threshold"] = calc_thresholds["upper_threshold"].replace( np.nan, 999999)
calc_thresholds["tax_year_id"] = calc_thresholds["tax_year_start"].str[:2].astype(int)

calc_thresholds["new_index_plus"]=calc_thresholds["new_index"] + 1

calc_thresholds = pd.merge( calc_thresholds, calc_thresholds[["new_index", "source", "level","tax_year_id"]],left_on=("new_index_plus", "source", "level"),right_on=("new_index", "source", "level"),how="left")
calc_thresholds =calc_thresholds[["tax_year_start","lower_threshold","upper_threshold", "rate","confirmed","source","level", "tax_year_id_x","tax_year_id_y"]]
calc_thresholds["tax_year_id_y"] = calc_thresholds["tax_year_id_y"].replace( np.nan, max_tax_year_id)
calc_thresholds = calc_thresholds.rename(columns={"tax_year_id_y": "tax_year_id_end","tax_year_id_x": "tax_year_id_start"})
calc_thresholds["tax_year_id_end"] = calc_thresholds["tax_year_id_end"].astype(int) - 1
calc_thresholds["tax_year_id"] = calc_thresholds.apply(
    lambda row: list(range(row["tax_year_id_start"], row["tax_year_id_end"] + 1)), axis=1
)

calc_thresholds = calc_thresholds.explode("tax_year_id").reset_index(drop=True)
calc_thresholds = calc_thresholds[["source","level","lower_threshold","upper_threshold","tax_year_id", "rate"]]
calc_thresholds = pd.merge( calc_thresholds, tax_years[["tax_year_id","tax_year_name"]], left_on="tax_year_id", right_on="tax_year_id",how="left")
calc_thresholds = calc_thresholds[["source","level","lower_threshold","upper_threshold","tax_year_id", "rate","tax_year_name"]]


calc_thresholds.to_csv('deductions/deduction_thresholds.csv', index= True)

# Confirmed Tax Payments

In [224]:
columns = ["tax_year", "deductions_for_year", "next_year_estimate", "jan_date","jan_payment", "jul_date", "jul_payment","confirmed"]

data = []

data.append(['24/25',	3622.9,	3173.9,	'30/12/2025'	,5209.85,	'31/03/2026'	,1586.95,	True])

confirmed_tax = pd.DataFrame(data, columns=columns)

confirmed_tax = pd.DataFrame(data, columns=columns)
